In [506]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

## Part zero configuration

In [507]:
ROOT_DIR=Path.cwd().parent.parent.parent
DATASET_DIR= ROOT_DIR / "dataset"
CSV_FILE_PATH = DATASET_DIR / "raw_loan_dataset.csv"
OUT_DIR=Path().cwd() / "data"
OUT_CSV_FILE= OUT_DIR / "clean_loan_dataset.csv"
OUT_TRAIN_CSV_FILE = OUT_DIR / "train.csv"
OUT_TEST_CSV_FILE = OUT_DIR / "test.csv"
print(ROOT_DIR)
print(DATASET_DIR)
print(CSV_FILE_PATH)
print(OUT_DIR)
print(OUT_CSV_FILE)
print(OUT_TRAIN_CSV_FILE)
print(OUT_TEST_CSV_FILE)

/home/mdev/Documents/ml/ds-ml-bootcamp
/home/mdev/Documents/ml/ds-ml-bootcamp/dataset
/home/mdev/Documents/ml/ds-ml-bootcamp/dataset/raw_loan_dataset.csv
/home/mdev/Documents/ml/ds-ml-bootcamp/submissions/marshaale/05_assignment/data
/home/mdev/Documents/ml/ds-ml-bootcamp/submissions/marshaale/05_assignment/data/clean_loan_dataset.csv
/home/mdev/Documents/ml/ds-ml-bootcamp/submissions/marshaale/05_assignment/data/train.csv
/home/mdev/Documents/ml/ds-ml-bootcamp/submissions/marshaale/05_assignment/data/test.csv


## Load & Inspect

In [508]:
df = pd.read_csv(CSV_FILE_PATH)
df = df.rename(columns={'Approved':'Status'})
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved


In [509]:
rows,columns = df.shape
print("ROWS:",rows)
print("COLUMNS:",columns)

ROWS: 103
COLUMNS: 7


In [510]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB


In [511]:
df.isna().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

In [512]:
def display_unique_values(col:str):
    print(f"\n{col}:",df[col].unique())

In [513]:
categorical_cols = ['Status','HasCollateral','PreviousDefaults']
for col in categorical_cols:
    display_unique_values(col) # use to capture and understand typo errors and in consistence category


Status: ['No' 'Yes' 'Approved' 'Rejected' 'approved' 'rejected' 'YES' 'NO']

HasCollateral: ['Yes' 'Y' 'No' 'N' nan 'yse' 'yes']

PreviousDefaults: ['No' nan 'Yes' '1' '0' 'Y' 'N']


## Clean Currency Formatting

In [514]:
def clean_numeric_col(col:str):
    print(f"\nCleaning column: {col}")
    df[col] = df[col].replace(r'[^0-9.]','',regex=True).astype(str)
    df[col] = df[col].str.strip()
    df[col] = df[col].replace('',np.nan)
    df[col] = pd.to_numeric(df[col],errors="coerce")

In [515]:
currency_cols = ['Income','LoanAmount']
for col in currency_cols:
    clean_numeric_col(col)
    print(df[col].head())


Cleaning column: Income
0    108810.0
1     96482.0
2     28478.0
3     25851.0
4     38396.0
Name: Income, dtype: float64

Cleaning column: LoanAmount
0    25800.0
1    11200.0
2    12100.0
3     7000.0
4    10700.0
Name: LoanAmount, dtype: float64


## Fix Category Errors before Imputation

In [516]:
categorical_map = {
    '1':'Yes','Approved':'Yes','approved':'Yes','Y':'Yes','yse':'Yes',
    'Rejected':'No','rejected':'No','N':'No','0':'No'
    
}

def clean_format_categorical_col(col:str,categorical_map:dict):
    print(f"\nCleaning column: {col}")
    df[col] = df[col].replace(categorical_map).astype(str)
    df[col] = df[col].str.strip().str.title()
    df[col] = df[col].replace({'Nan':np.nan})

In [517]:
for col in categorical_cols:
    display_unique_values(col)
    clean_format_categorical_col(col,categorical_map)
    display_unique_values(col)


Status: ['No' 'Yes' 'Approved' 'Rejected' 'approved' 'rejected' 'YES' 'NO']

Cleaning column: Status

Status: ['No' 'Yes']

HasCollateral: ['Yes' 'Y' 'No' 'N' nan 'yse' 'yes']

Cleaning column: HasCollateral

HasCollateral: ['Yes' 'No' nan]

PreviousDefaults: ['No' nan 'Yes' '1' '0' 'Y' 'N']

Cleaning column: PreviousDefaults

PreviousDefaults: ['No' nan 'Yes']


## Impute Missing Values

In [518]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     float64
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     float64
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(4), object(3)
memory usage: 5.8+ KB


In [519]:
df.isna().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

In [520]:
def fill_with_median(col:str):
    print(f"\nFilling column: {col}")
    df[col] = df[col].fillna(df[col].median())
        
def fill_with_mode(col:str):
    print(f"\nFilling column: {col}")
    df[col] = df[col].fillna(df[col].mode()[0])

In [521]:
numeric_cols = currency_cols + ['CreditScore','EmploymentYears']
print(currency_cols)
print(numeric_cols)

['Income', 'LoanAmount']
['Income', 'LoanAmount', 'CreditScore', 'EmploymentYears']


In [522]:
for col in numeric_cols:
    fill_with_median(col)


Filling column: Income

Filling column: LoanAmount

Filling column: CreditScore

Filling column: EmploymentYears


In [523]:
for col in categorical_cols:
    fill_with_mode(col)
    display_unique_values(col)


Filling column: Status

Status: ['No' 'Yes']

Filling column: HasCollateral

HasCollateral: ['Yes' 'No']

Filling column: PreviousDefaults

PreviousDefaults: ['No' 'Yes']


In [524]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            103 non-null    float64
 1   CreditScore       103 non-null    float64
 2   EmploymentYears   103 non-null    float64
 3   LoanAmount        103 non-null    float64
 4   HasCollateral     103 non-null    object 
 5   PreviousDefaults  103 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(4), object(3)
memory usage: 5.8+ KB


In [525]:
df.isna().sum()

Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Status              0
dtype: int64

## Remove Duplicates

In [526]:
before = df.shape
df = df.drop_duplicates()
print(f"Before dropping {before}(rows,cols) after dropped {df.shape}(rows,cols)")

Before dropping (103, 7)(rows,cols) after dropped (100, 7)(rows,cols)


## Outliers (IQR capping)

In [527]:
def iqr_fun(col:pd.Series,k=1.5):
    q1,q3 = col.quantile([0.25,0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

In [528]:
for col in numeric_cols:
    print(f"\nCapping column: {col}")
    lower,upper = iqr_fun(df[col])
    print("lower and upper before capping:(lower,upper)",(df[col].min(),df[col].max()))
    print("lower and upper of iqr:(lower,upper)",(lower,upper))
    df[col] = df[col].clip(lower=lower,upper=upper)
    print("lower and upper after capping:(lower,upper)",(df[col].min(),df[col].max()))


Capping column: Income
lower and upper before capping:(lower,upper) (25851.0, 250000.0)
lower and upper of iqr:(lower,upper) (-23827.875, 167619.125)
lower and upper after capping:(lower,upper) (25851.0, 167619.125)

Capping column: LoanAmount
lower and upper before capping:(lower,upper) (5000.0, 180000.0)
lower and upper of iqr:(lower,upper) (-16687.5, 66212.5)
lower and upper after capping:(lower,upper) (5000.0, 66212.5)

Capping column: CreditScore
lower and upper before capping:(lower,upper) (484.0, 920.0)
lower and upper of iqr:(lower,upper) (344.25, 962.25)
lower and upper after capping:(lower,upper) (484.0, 920.0)

Capping column: EmploymentYears
lower and upper before capping:(lower,upper) (0.5, 25.0)
lower and upper of iqr:(lower,upper) (-11.275, 35.525000000000006)
lower and upper after capping:(lower,upper) (0.5, 25.0)


## Label Encoding

In [529]:
for col in categorical_cols:
    print(f"\nEncoding column: {col}")
    df[col] = df[col].map({"Yes":1,"No":0}).astype(int)


Encoding column: Status

Encoding column: HasCollateral

Encoding column: PreviousDefaults


In [530]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            100 non-null    float64
 1   CreditScore       100 non-null    float64
 2   EmploymentYears   100 non-null    float64
 3   LoanAmount        100 non-null    float64
 4   HasCollateral     100 non-null    int64  
 5   PreviousDefaults  100 non-null    int64  
 6   Status            100 non-null    int64  
dtypes: float64(4), int64(3)
memory usage: 6.2 KB


In [531]:
df[categorical_cols].head()

,Status,HasCollateral,PreviousDefaults
0,0,1,0
1,1,1,0
2,1,0,0
3,1,1,0
4,1,0,0


## Class Balance Check

In [532]:
class_ratio = df['Status'].value_counts(normalize=True)
class_ratio

Status
1    0.66
0    0.34
Name: proportion, dtype: float64

In [533]:
if class_ratio.max() > 0.7:
    print("\nWarning: severely imbalanced classes")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")


Class balance OK for baseline Accuracy (both classes well represented).


## Feature Engineering (no leakage)

In [534]:
df['IncomePerYearEmployed'] = df['Income'] / df['EmploymentYears'].replace(0,np.nan)
df['IncomePerYearEmployed'].isna().sum()

np.int64(0)

In [535]:
df['IncomePerYearEmployed']

0     98918.181818
1      6432.133333
2      5273.703704
3      1468.806818
4      3917.959184
          ...     
95     5287.569721
96     2509.877049
97     6863.333333
98     5541.420455
99     5103.934426
Name: IncomePerYearEmployed, Length: 100, dtype: float64

In [536]:
df['DebtToIncome'] = (df['LoanAmount'] / df['Income'].replace(0,np.nan)) * 100
df['DebtToIncome'].isna().sum()

np.int64(0)

In [537]:
df['DebtToIncome']

0     23.711056
1     11.608383
2     42.488939
3     27.078256
4     27.867486
        ...    
95    57.565666
96    36.250225
97    25.066976
98    16.302843
99    33.243399
Name: DebtToIncome, Length: 100, dtype: float64

## Feature Scaling

In [538]:
scaler = RobustScaler()

In [539]:
train, test = train_test_split(df, test_size=0.2, random_state=42)

In [540]:
train.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status,IncomePerYearEmployed,DebtToIncome
55,66114.0,703.0,15.4,23500.0,0,1,0,4293.116883,35.544665
88,69460.5,622.0,22.1,8100.0,0,0,1,3143.009050,11.661304
26,78354.0,577.0,2.8,33700.0,0,0,1,27983.571429,43.009929
42,45374.0,619.0,7.4,9300.0,0,0,1,6131.621622,20.496319
69,35155.0,626.0,6.2,9400.0,1,0,1,5670.161290,26.738728


In [541]:
test.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status,IncomePerYearEmployed,DebtToIncome
83,102607.0,801.0,6.4,46000.0,0,1,0,16032.343750,44.831249
53,61585.0,659.0,16.2,18500.0,0,1,0,3801.543210,30.039782
70,115957.0,558.0,17.8,66212.5,0,0,1,6514.438202,57.100908
45,117493.0,920.0,14.2,18300.0,1,0,1,8274.154930,15.575396
44,61265.0,502.0,0.6,29700.0,0,0,0,102108.333333,48.477924


In [542]:
cols = df.select_dtypes(include=['int64','float64']).columns.to_list()
to_scale_cols = [ x for x in cols if x not in categorical_cols ]
print(cols)
print(to_scale_cols)

['Income', 'CreditScore', 'EmploymentYears', 'LoanAmount', 'HasCollateral', 'PreviousDefaults', 'Status', 'IncomePerYearEmployed', 'DebtToIncome']
['Income', 'CreditScore', 'EmploymentYears', 'LoanAmount', 'IncomePerYearEmployed', 'DebtToIncome']


In [543]:
train[to_scale_cols] = scaler.fit_transform(train[to_scale_cols])
test[to_scale_cols] = scaler.transform(test[to_scale_cols])
df[to_scale_cols] = scaler.fit_transform(df[to_scale_cols])

In [544]:
train.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status,IncomePerYearEmployed,DebtToIncome
55,-0.070256,0.521042,0.252747,0.134132,0,1,0,-0.281279,0.128642
88,0.001583,-0.128257,0.841758,-0.603593,0,0,1,-0.501191,-0.881289
26,0.192500,-0.488978,-0.854945,0.622754,0,0,1,4.248573,0.444317
42,-0.515483,-0.152305,-0.450549,-0.546108,0,0,1,0.070262,-0.507692
69,-0.734855,-0.096192,-0.556044,-0.541317,1,0,1,-0.017974,-0.243726


In [545]:
test.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status,IncomePerYearEmployed,DebtToIncome
83,0.713141,1.306613,-0.538462,1.211976,0,1,0,1.963379,0.521334
53,-0.167481,0.168337,0.323077,-0.105389,0,1,0,-0.375273,-0.104138
70,0.999726,-0.641283,0.463736,2.180240,0,0,1,0.143460,1.040168
45,1.032700,2.260521,0.147253,-0.114970,1,0,1,0.479935,-0.715778
44,-0.174350,-1.090180,-1.048352,0.431138,0,0,0,18.421969,0.675537


In [546]:
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status,IncomePerYearEmployed,DebtToIncome
0,0.822149,-0.653722,-0.978632,0.246080,1,0,0,14.662142,-0.445244
1,0.564574,-0.737864,0.209402,-0.458384,1,0,1,0.129843,-0.912749
2,-0.856268,0.000000,-0.611111,-0.414958,0,0,1,-0.052181,0.280113
3,-0.911156,-0.498382,0.431624,-0.661037,1,0,1,-0.650043,-0.315175
4,-0.649046,-0.718447,-0.235043,-0.482509,0,0,1,-0.265208,-0.284688


**Which scaler did you choose and why does it fit this loan dataset?**

I chose `RobustScaler` because it handles outliers well and centers the features so the median becomes 0. Since this is a loan approval 

dataset with numeric features, using `RobustScaler` helps reduce the impact of extreme values while keeping the data properly scaled.

[Mastering Data Preprocessing: Simplified Guide to Feature Scaling and Transformations in Machine Learning](https://medium.com/@ehsan.sepahsalari/mastering-data-preprocessing-simplified-guide-to-feature-scaling-and-transformations-in-machine-7ea5e17e5b82)

## Final Checks & Save

In [547]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int64  
 5   PreviousDefaults       100 non-null    int64  
 6   Status                 100 non-null    int64  
 7   IncomePerYearEmployed  100 non-null    float64
 8   DebtToIncome           100 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.8 KB


In [548]:
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status,IncomePerYearEmployed,DebtToIncome
0,0.822149,-0.653722,-0.978632,0.246080,1,0,0,14.662142,-0.445244
1,0.564574,-0.737864,0.209402,-0.458384,1,0,1,0.129843,-0.912749
2,-0.856268,0.000000,-0.611111,-0.414958,0,0,1,-0.052181,0.280113
3,-0.911156,-0.498382,0.431624,-0.661037,1,0,1,-0.650043,-0.315175
4,-0.649046,-0.718447,-0.235043,-0.482509,0,0,1,-0.265208,-0.284688


In [549]:
df.isna().sum()

Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Status                   0
IncomePerYearEmployed    0
DebtToIncome             0
dtype: int64

In [550]:
print(train.info())
print(train.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 80 entries, 55 to 51
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 80 non-null     float64
 1   CreditScore            80 non-null     float64
 2   EmploymentYears        80 non-null     float64
 3   LoanAmount             80 non-null     float64
 4   HasCollateral          80 non-null     int64  
 5   PreviousDefaults       80 non-null     int64  
 6   Status                 80 non-null     int64  
 7   IncomePerYearEmployed  80 non-null     float64
 8   DebtToIncome           80 non-null     float64
dtypes: float64(6), int64(3)
memory usage: 6.2 KB
None
Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Status                   0
IncomePerYearEmployed    0
DebtToIncome             0
dtype: int64


In [551]:
print(test.info())
print(test.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 20 entries, 83 to 31
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 20 non-null     float64
 1   CreditScore            20 non-null     float64
 2   EmploymentYears        20 non-null     float64
 3   LoanAmount             20 non-null     float64
 4   HasCollateral          20 non-null     int64  
 5   PreviousDefaults       20 non-null     int64  
 6   Status                 20 non-null     int64  
 7   IncomePerYearEmployed  20 non-null     float64
 8   DebtToIncome           20 non-null     float64
dtypes: float64(6), int64(3)
memory usage: 1.6 KB
None
Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Status                   0
IncomePerYearEmployed    0
DebtToIncome             0
dtype: int64


In [552]:
Path.mkdir(OUT_DIR,exist_ok=True)

In [553]:
df.to_csv(OUT_CSV_FILE,index=False)
train.to_csv(OUT_TRAIN_CSV_FILE,index=False)
test.to_csv(OUT_TEST_CSV_FILE,index=False)